# Ejemplo Práctico: Ponderación Objetiva de Criterios
### Materia: Herramientas Multicriterio para Toma de Decisiones (Doctorado)
**Tema:** Cálculo de Pesos por Entropía de Shannon y Método CRITIC en Python
**Profesor:** Dr. Leonardo Gabriel Hernández Landa

Este notebook contiene la resolución completa paso a paso del caso de estudio de evaluación de 10 transportistas terrestres consolidados (LTL) bajo 4 criterios operativos, utilizando métodos de ponderación objetiva.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

## 1. Definición del Dataset
Cargamos la matriz de decisión original $X_{10 \times 4}$ en un DataFrame de pandas. 

Los criterios y sus direcciones son:
*   **Costo ($C_1$):** Minimizar ($MXN$)
*   **OTIF ($C_2$):** Maximizar ($\%$)
*   **Lead Time ($C_3$):** Minimizar ($horas$)
*   **Siniestralidad ($C_4$):** Minimizar ($\%$)

In [ ]:
# Crear dataset de transportistas
data = {
    'Costo': [12500, 10200, 14000, 11000, 13200, 9800, 15500, 12000, 11500, 10800],
    'OTIF': [0.95, 0.88, 0.98, 0.91, 0.96, 0.85, 0.99, 0.94, 0.92, 0.89],
    'Lead_Time': [24.0, 28.0, 20.0, 26.0, 22.0, 30.0, 18.0, 24.0, 25.0, 27.0],
    'Siniestralidad': [0.005, 0.012, 0.002, 0.008, 0.004, 0.015, 0.001, 0.006, 0.007, 0.010]
}
index = [f'A{i}' for i in range(1, 11)]
df_original = pd.DataFrame(data, index=index)
df_original

## 2. Ponderación por Entropía de Shannon

El algoritmo de Entropía requiere los siguientes pasos:
1.  **Alineación de direcciones:** Los criterios de costo (minimizar) deben invertirse ($x'_{ij} = 1/x_{ij}$).
2.  **Normalización Sumatoria:** $p_{ij} = x'_{ij} / \sum x'_{kj}$.
3.  **Cálculo de Entropía:** $e_j = -k \sum p_{ij} \ln(p_{ij})$ donde $k = 1/\ln(m)$ (con $m = 10$).
4.  **Grado de Divergencia:** $d_j = 1 - e_j$.
5.  **Pesos Finales:** $w_j = d_j / \sum d_k$.

In [ ]:
# Paso 1: Alinear direcciones (inversión de costo)
df_entropy_aligned = df_original.copy()
for col in ['Costo', 'Lead_Time', 'Siniestralidad']:
    df_entropy_aligned[col] = 1.0 / df_entropy_aligned[col]

# Paso 2: Normalización sumatoria
df_p = df_entropy_aligned / df_entropy_aligned.sum()

# Paso 3: Cálculo de la Entropía de Shannon
m = len(df_original)  # m = 10 alternativas
k = 1.0 / np.log(m)

# Función para calcular p * ln(p) manejando p = 0
def p_log_p(p):
    return np.where(p > 0, p * np.log(p), 0.0)

entropias = -k * df_p.apply(p_log_p).sum()

# Paso 4: Grado de Divergencia
divergencias = 1.0 - entropias

# Paso 5: Pesos Finales por Entropía
w_entropy = divergencias / divergencias.sum()

print("--- Resultados de Entropía de Shannon ---")
for col, w, e in zip(df_original.columns, w_entropy, entropias):
    print(f"Criterio {col:<15} | Entropía: {e:.5f} | Peso: {w*100:.2f}%")

## 3. Ponderación por el Método CRITIC

El algoritmo CRITIC sigue estos pasos:
1.  **Normalización Max-Min:**
    *   Beneficio: $r_{ij} = \frac{x_{ij} - x_j^{\min}}{x_j^{\max} - x_j^{\min}}$
    *   Costo: $r_{ij} = \frac{x_j^{\max} - x_{ij}}{x_j^{\max} - x_j^{\min}}$
2.  **Cálculo de Desviación Estándar** (columna de la matriz normalizada $\mathbf{R}$).
3.  **Cálculo de la Matriz de Correlación de Pearson ($r_{jk}$)**.
4.  **Cálculo de Cantidad de Información ($C_j$):** $C_j = \sigma_j \sum_{k=1}^n (1 - r_{jk})$.
5.  **Pesos Finales:** $w_j = C_j / \sum C_k$.

In [ ]:
# Paso 1: Normalización Max-Min
df_critic_norm = df_original.copy()
for col in df_original.columns:
    val_max = df_original[col].max()
    val_min = df_original[col].min()
    if col in ['Costo', 'Lead_Time', 'Siniestralidad']:  # Criterios de costo (minimizar)
        df_critic_norm[col] = (val_max - df_original[col]) / (val_max - val_min)
    else:  # Criterios de beneficio (maximizar)
        df_critic_norm[col] = (df_original[col] - val_min) / (val_max - val_min)

# Paso 2: Desviación estándar de cada criterio
desviaciones = df_critic_norm.std(ddof=0)  # Desviación estándar poblacional

# Paso 3: Matriz de correlación de Pearson
matriz_corr = df_critic_norm.corr(method='pearson')

# Paso 4: Cantidad de información C_j
conflictos = (1.0 - matriz_corr).sum(axis=0)
C_j = desviaciones * conflictos

# Paso 5: Pesos Finales por CRITIC
w_critic = C_j / C_j.sum()

print("--- Resultados de CRITIC ---")
for col, w, std, c_j in zip(df_original.columns, w_critic, desviaciones, C_j):
    print(f"Criterio {col:<15} | Desv. Est.: {std:.5f} | Inf. (C_j): {c_j:.5f} | Peso: {w*100:.2f}%")

## 4. Visualización y Análisis Comparativo

Graficamos el mapa de calor de correlaciones entre criterios y comparamos el perfil de pesos que arroja cada método.

In [ ]:
# Graficar mapa de calor de correlaciones
plt.figure(figsize=(7, 5))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".3f")
plt.title('Mapa de Calor de Correlaciones de Pearson (Matriz Normalizada)')
plt.tight_layout()
plt.show()

In [ ]:
# Comparación de perfiles de pesos
df_pesos = pd.DataFrame({
    'Entropía': w_entropy,
    'CRITIC': w_critic
}, index=df_original.columns)

df_pesos_plot = df_pesos.reset_index().melt(id_vars='index', var_name='Método', value_name='Peso')
df_pesos_plot.columns = ['Criterio', 'Método', 'Peso']

plt.figure(figsize=(10, 6))
sns.barplot(data=df_pesos_plot, x='Criterio', y='Peso', hue='Método', palette='Set2')
plt.ylabel('Peso Relativo')
plt.title('Comparación de Perfiles de Pesos: Entropía de Shannon vs. CRITIC')
plt.ylim(0, 0.6)

# Mostrar porcentajes en las barras
ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height*100:.1f}%',
                    (p.get_x() + p.get_width() / 2., height + 0.01),
                    ha='center', va='center', 
                    xytext=(0, 5), 
                    textcoords='offset points', 
                    fontsize=10)

plt.tight_layout()
plt.show()

## 5. Análisis Estadístico y Conclusión
Comparamos la similitud de los rankings mediante correlación de Spearman.

In [ ]:
# Correlación de Spearman entre pesos
corr_spearman = df_pesos['Entropía'].corr(df_pesos['CRITIC'], method='spearman')
print(f"Correlación de Spearman entre los perfiles de pesos de Entropía y CRITIC: {corr_spearman:.4f}")

print("\n--- Análisis comparativo de pesos finales ---")
df_pesos['Diferencia_Absoluta'] = (df_pesos['Entropía'] - df_pesos['CRITIC']).abs()
print(df_pesos)